# DEP Invasive Species Random Forest — Beginner Step-by-Step Notebook

## 5-Day Participant Learning Notebook

This notebook takes you from **basic Python and Jupyter skills** into the same invasive-species Random Forest workflow used in the original modelling notebook.

### Learning pathway
**Python basics → field data → AOI → Sentinel-2 → cloud masking → NDVI → training samples → Random Forest → 80/20 accuracy → confusion matrix → final map → new AOI**

### Important workshop rule
You do **not** need to memorise Python. Focus on:
1. what each stage does;
2. what information goes into the model;
3. what the output means;
4. how to recognise when something is wrong; and
5. how field observations affect the final classification.

Run cells from top to bottom using **Shift + Enter**.

## How participant exercises work

This version intentionally does **not** give you every answer.

Look for three types of activity:

### 🟢 TRY IT
Small edits where you change a value or complete one line.

### 🟡 THINK FIRST
Predict what the code/output will do before you run it.

### 🔵 CHALLENGE
A partially completed task. Work with your group to finish it using earlier examples in the notebook.

If you get stuck:
1. read the error message;
2. compare with an earlier working cell;
3. check spelling, brackets, quotes and variable names;
4. ask another participant;
5. then ask the facilitator.

The aim is to understand the workflow, not simply press **Run**.

# Part 0 — Getting comfortable with Jupyter and Python

Before working with satellite imagery, spend a little time learning the language used in the notebook.

## Jupyter terms

- **Notebook** — this whole `.ipynb` file.
- **Cell** — one block of text or Python code.
- **Markdown cell** — explanations and instructions.
- **Code cell** — Python instructions that can be executed.
- **Kernel** — the Python session running behind the notebook.
- **Output** — what Python returns after a cell is run.
- **Error/Traceback** — Python explaining where something went wrong.

### Beginner Activity 1 — Run your first Python cell
Run the cell below with **Shift + Enter**.

In [ ]:
print("Bula! My first Python cell is working.")

### Beginner Activity 2 — Variables

A **variable** is a name used to store information.

Change the values below to your own workshop details and run the cell again.

In [ ]:
participant_name = "Your name"
country = "Vanuatu"
workshop_day = 1

print("Participant:", participant_name)
print("Country:", country)
print("Workshop day:", workshop_day)

### Beginner Activity 3 — Numbers and calculations

Python can be used like a calculator. Here we calculate a percentage.

**Try it yourself:** change `total_points` and `correct_points`.

In [ ]:
total_points = 100
correct_points = 82

accuracy_percent = (correct_points / total_points) * 100

print("Accuracy:", accuracy_percent, "%")

### Beginner Activity 4 — Lists and indexing

A **list** stores several items together. Python starts counting positions at **0**.

The first item is therefore `species[0]`.

In [ ]:
species = [
    "African Tulip",
    "Cordia alliodora",
    "Merremia peltata"
]

print("Whole list:", species)
print("First species:", species[0])
print("Second species:", species[1])
print("Number of species:", len(species))

### Beginner Activity 5 — Dictionaries

A **dictionary** stores information as `key: value` pairs. This is useful for class codes and class names.

In [ ]:
class_names = {
    1: "African Tulip",
    2: "Cordia alliodora",
    3: "Merremia peltata"
}

print(class_names)
print("Class 1 is:", class_names[1])

### Beginner Activity 6 — A simple `if` statement

An `if` statement lets Python make a decision.

Try changing `cloud_percent` to `15`, then to `70`.

In [ ]:
cloud_percent = 25

if cloud_percent <= 40:
    print("Scene passes our example cloud threshold.")
else:
    print("Scene is too cloudy for this example.")

### Beginner Activity 7 — A `for` loop

A loop repeats an instruction for each item in a collection.

In [ ]:
bands = ["red", "green", "blue", "nir08", "swir16"]

for band in bands:
    print("Satellite predictor:", band)

### Beginner Activity 8 — Importing packages

Python packages contain tools written by other developers.

In the modelling workflow you will see packages such as:

- `numpy` — numerical arrays;
- `pandas` — tables;
- `xarray` — multidimensional satellite data;
- `geopandas` — spatial vector data;
- `matplotlib` — plots;
- `sklearn` — machine-learning tools.

Run the imports below.

In [ ]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

### Beginner Activity 9 — Build a tiny table

A **DataFrame** is similar to a spreadsheet or attribute table.

This miniature example represents field observations.

In [ ]:
practice_data = pd.DataFrame({
    "species": ["African Tulip", "Cordia", "Forest", "Grassland"],
    "class_code": [1, 2, 8, 6],
    "gps_accuracy_m": [2.1, 4.5, 3.2, 5.0]
})

practice_data

### Beginner Activity 10 — Select a column and filter rows

This introduces the same kind of table operations used later when preparing model training data.

In [ ]:
print("Class codes:")
print(practice_data["class_code"])

good_gps = practice_data[
    practice_data["gps_accuracy_m"] <= 4
]

print("\nObservations with GPS accuracy ≤ 4 m:")
display(good_gps)

## Beginner checkpoint

Before moving on, you should be able to explain:

- What is a variable?
- What is a list?
- What is a DataFrame?
- What does `print()` do?
- What is the difference between a Markdown cell and a code cell?
- What does an error message mean?

You do not need to memorise the syntax.

---

# Part 1 — From Python to Earth Observation

Now we move into the real invasive-species workflow.

## Key modelling terms

| Term | Meaning in this workshop |
|---|---|
| **AOI** | Area of Interest — the area we want to analyse |
| **Field observation** | A labelled location collected on the ground |
| **Class** | The land-cover/species category assigned to an observation |
| **Predictor** | Satellite information supplied to the model |
| **Band** | A wavelength range measured by Sentinel-2 |
| **NDVI** | Vegetation index derived from red and near-infrared reflectance |
| **SCL** | Sentinel-2 Scene Classification Layer used to help mask unsuitable pixels |
| **Composite** | Multiple satellite observations summarised into one representative image |
| **Training data** | Observations used by the model to learn patterns |
| **Test data** | Observations withheld from training and used to assess performance |
| **Random Forest** | A machine-learning method made from many decision trees |
| **Confusion matrix** | Table showing correct classifications and class-to-class mistakes |
| **Precision** | Of pixels/points predicted as a class, how many were correct? |
| **Recall** | Of the true observations of a class, how many did the model find? |
| **F1-score** | Balance between precision and recall |

### Think before coding
Why might an African Tulip observation and a native forest observation look similar to a satellite sensor?

Discuss with the person beside you before continuing.

# Machine Learning using Sentinel-2 Data

<div><img src="https://github.com/digitalearthpacific/DigitalEarthPacific/blob/main/notebooks/DEP_LCAST/reference_materials/images/1B/1.a.png?raw=true" width="500"/></div>

This example uses training data from the
[Coast Train](https://github.com/nick-murray/coastTrain) dataset
along with Sentinel-2 data to demonstrate how to use a
machine learning classifier, in this case, Random Forest, to
assign a class to each pixel.

This notebook combines lessons from previous notebooks into
a comprehensive worked example.

## Getting started

First we load the required Python libraries and tools.

### 🟢 TRY IT — Imports
Before running the setup, name **three Python packages** you expect to use and what each one does.

**Your answer:**
- Package 1:
- Package 2:
- Package 3:

# Part 2 — Set up the modelling environment

Import the packages required by the real workflow. If an import fails, read the last line of the error first.

In [ ]:
from pystac_client import Client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import folium
import joblib
import matplotlib.pyplot as plt


from sklearn.ensemble import RandomForestClassifier

import odc.geo.xr  # noqa: F401

## Study site configuration

Here we establish the STAC catalog we're using as well as a
spatial and temporal extent. This can be anywhere, but this location
near Kuching was chosen due to the training data having several
classes available.

### 🟡 THINK FIRST — Satellite catalogue
What information do you think the catalogue search needs before it can find Sentinel-2 imagery?

Write at least three things:
1.
2.
3.

# Part 5 — Connect to the satellite data catalogue

The notebook searches a STAC catalogue for Sentinel-2 imagery covering the AOI.

In [ ]:
# STAC Catalog URL
# catalog = "https://stac.staging.digitalearthpacific.org"
catalog = "https://earth-search.aws.element84.com/v1"
# Create a STAC Client
client = Client.open(catalog)

<font color='blue'>1.1. Define your area of interest. Find the coordinates of the bottom left and top right corners of your bounding box / area of interest. 

Use QGIS, Google Maps or another site to find the coordinates. Make sure to use at least 4 or 5 decimal places. Lat = latitude and lon = longitude. The min is in the bottom left and max is in the top right.   

### 🟡 THINK FIRST — AOI
Before running the AOI cell:

1. What does **AOI** stand for?
2. What would happen if the AOI were accidentally placed in the wrong country?
3. Which coordinate reference system do you expect the AOI to use at this stage?

Write your prediction here before running the code.

# Part 3 — Define the Area of Interest (AOI)

Here we define where Sentinel-2 data and field observations will be analysed.

**Checkpoint:** Can you identify the AOI coordinates or geometry being used?

In [ ]:
# # 1.1 - input your four coordinates here 
# min_lat = -17.631319
# min_lon = 177.968450
# max_lat = -17.621633
# max_lon = 177.975529

# bbox = (min_lon, min_lat, max_lon, max_lat)

### 🔵 CHALLENGE — Inspect the field data
Using the examples from the Python introduction, write code in the empty cell below to:

1. print the number of field observations;
2. display the first five rows;
3. print the available column names.

Do this **before** looking at later cells for hints.

In [ ]:
# PARTICIPANT CHALLENGE
# Write your field-data inspection code below.
# Hint: useful Python ideas include len(), .head() and .columns.

# Part 4 — Load the field observations

Field observations provide the labelled examples that teach the Random Forest.

**Participant activity:** inspect the attribute columns and identify the class field.

In [ ]:
# Define bounding box – should match your area
aoi_gdf = gpd.read_file("AOI/Lololo_2.geojson")
aoi_gdf.plot(edgecolor="red", facecolor="none")
plt.title("AOI Check")
plt.show()  # Replace with actual coordinates

In [ ]:
bbox = aoi_gdf.total_bounds  # returns [minx, miny, maxx, maxy]

In [ ]:
center = [
    aoi_gdf.geometry.centroid.y.mean(),
    aoi_gdf.geometry.centroid.x.mean()
]

m = folium.Map(location=center, zoom_start=10)

folium.GeoJson(
    aoi_gdf,
    style_function=lambda x: {
        "fillColor": "none",
        "color": "red",
        "weight": 2
    }
).add_to(m)

m

In [ ]:
min_lat = aoi_gdf['geometry'].bounds['miny'].min()
max_lat = aoi_gdf['geometry'].bounds['maxy'].max()
min_lon = aoi_gdf['geometry'].bounds['minx'].min()
max_lon = aoi_gdf['geometry'].bounds['maxx'].max()

<font color='blue'>1.2. Define the time period of interest. 

You can now define the time you are interested in. It is goood to put the format as "year-month"/"year-month". For example "2022-06/2024-09" to get all images covering June 2022 to September 2024. You can try to just choose the flowering months in one year but may have issues with cloud cover. Try different time periods and see what happes. 

In [ ]:
# 1.2 - input your datetime here - recommend at least 3 months and max 3 years. 
datetime = "2025-06/2025-12"

In [ ]:
# Create local dask cluster to improve data load time. Only run this once.
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='16GB')

# Configure S3 access. Cloud defaults is an optimisation, while requester pays is required for Landsat
configure_s3_access(cloud_defaults=True, requester_pays=True)

## Training data

Next up we gather training data. This could be any geospatial point dataset
with a column that is numeric, for the class.

If you'd like to explore the structure of this data, you can run `gdf.head()`
to see the first few rows. The `explore()` function with the `column` argument
will show the data on the map, and change the colour based on that column.

<font color='blue'>2.1. Input your data from the field.  

If you have new data, save your data as a geojson in QGIS and then drag and drop it into the same folder as this notebook in DEP. Then you will have to put the name of the file in the brackets below inside of quotes: eg. `'name.geojson'`   

In [ ]:
# 2.1 - input your data file here inside the '' 
gdf = gpd.read_file("Training_Data/Fiji_updated8.geojson", bbox=tuple(bbox))

# gdf = gdf.fillna(0)
gdf.explore(column="Random_Forest", legend=True)

In [ ]:
# # 2.1 - input your data file here inside the '' 
# gdf = gpd.read_file('InvasiveClean4.geojson', bbox=bbox)

# # gdf = gdf.fillna(0)
# gdf.explore(column="randomforest", legend=True)

## Find and load Sentinel-2 data

Here we search for Sentinel-2 scenes over our study area and use
Dask to lazy-load them. We're only loading the red, green, blue, nir and swir
bands, along with the scene classification (scl) band.

<font color='blue'>2.2. - define the satellite image collections you wish to use: 

Input your collections name inside the brackets: 

You may try:   
"sentinel-2-c1-l2a" (10mx10m resolution pixels)   
OR   
"landsat-c2-l2" (30mx30m resolution pixels)

<font color='blue'>2.3. - define your cloud cover threshold. We recommend a number somewhere between 10-50. No need for quotes. 

Write this next to "lt": eg. {"lt": 50}} 


### 🟢 TRY IT — Date and cloud experiment
Find the Sentinel-2 search settings in the next cells.

Before changing anything, record:
- current date range:
- current cloud threshold:

Later, try changing **one** of these and note how the number of available scenes changes.

# Part 6 — Search for Sentinel-2 scenes

Satellite imagery varies through time and cloud conditions. This step finds candidate scenes for the AOI.

In [ ]:
# Search for Sentinel-2 data
items = client.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=bbox,
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": 25 }},
).item_collection()

print(f"Found {len(items)} items")

In [ ]:
# Load the data into an xarray Dataset
data = load(
    items,
    measurements=["red", "green", "blue", "nir08", "swir16", "scl"],
    bbox=bbox,
    chunks={"x": 2048, "y": 2048},
    groupby="solar_day",
)

data

## Data preparation

Now that we have data, we need to clean it up, masking out clouds
and scaling values to between 0-1, which are the valid reflectance
values.

We add a couple of indices too, which will help the machine learning
algorithm.

Note that we still have a lazy-loaded array, and haven't transferred
any data over the network.

### 🔵 CHALLENGE — Why is SCL different?
Discuss with your group:

> Why should `scl` help us remove clouds but **not** be supplied as a Random Forest land-cover predictor?

Write a one- or two-sentence explanation before continuing.

# Part 8 — Cloud masking and spectral preparation

Cloud, shadow and unsuitable pixels can damage the training data. SCL is used here for masking; it is **not** used as a Random Forest predictor.

In [ ]:
# Mask out clouds and scale values

# Apply Sentinel-2 cloud mask
# 1: defective, 3: shadow, 9: high confidence cloud, 10: thin cirrus
mask_flags = [1, 3, 9, 10]

cloud_mask = ~data.scl.isin(mask_flags)
masked = data.where(cloud_mask)

# Apply scaling and clip to valid data, from 0 to 1
scaled = (masked.where(masked != 0) * 0.0001).clip(0, 1)

# Add some indices
scaled["ndvi"] = (scaled.nir08 - scaled.red) / (scaled.nir08 + scaled.red)
# scaled["ndwi"] = (scaled.green - scaled.nir08) / (scaled.green + scaled.nir08)

scaled


In [ ]:
# Visualise one date, to make sure it looks good.
# This example shows empty areas where we've masked out nodata, but
# note that there are still a lot of clouds coming in!

scaled.isel(time=0).odc.explore(vmin=0, vmax=0.3)

## Create a cloud-free composite

The final data preparation step involves creating a temporal
median of the data bands. Here we use `compute()` to process
the data and bring it into memory.

We preview the data in the second cell below.

### 🟡 THINK FIRST — Composite
Why might we use a temporal **median** rather than selecting only one Sentinel-2 scene?

Write two possible advantages:
1.
2.

# Part 10 — Create the satellite composite

A temporal median reduces the influence of individual noisy observations and creates a representative image.

In [ ]:
# Create a median composite, which should get rid of most of the remaining clouds
# Note that this will take a few minutes to complete

median = scaled.median("time").compute()

median

<font color='blue'>3.1. - visualise the resulting median satellite image:

`median.odc.explore(vmin=0, vmax=0.3)`

In [ ]:
median.odc.explore(vmin=0, vmax=0.3)

## Mask out Water Line, roadlines, and Settlements

In [ ]:
# Example arrays (replace with your own band data)
B03 = np.random.random((1000,1000))*1000   # Green (560 nm)
B04 = np.random.random((1000,1000))*1000   # Red (665 nm)
B08 = np.random.random((1000,1000))*1000   # NIR (842 nm)
B11 = np.random.random((1000,1000))*1000   # SWIR (1610 nm)


In [ ]:
np.seterr(divide='ignore', invalid='ignore')

# NDWI – Water
NDWI = (B03 - B08) / (B03 + B08)

# NDBI – Built-up
NDBI = (B11 - B08) / (B11 + B08)

# NDBaI – Bare soil / Roads
NDBaI = (B11 - B04) / (B11 + B04)


In [ ]:
# Typical thresholds
water_mask    = NDWI > 0.2
building_mask = NDBI > 0.1
road_mask     = (NDBaI > 0.15) & (NDWI < 0) & (NDBI < 0.2)


In [ ]:
combined_mask = water_mask | building_mask | road_mask


In [ ]:
if combined_mask.shape == median.shape:
    mask_da = xr.DataArray(combined_mask, coords=median.coords, dims=median.dims)
    masked_da = median.where(~mask_da, other=np.nan)

In [ ]:
masked_da

## Prepare training data array

This next step involves extracting observed values from the satellite data
and combining them with our point data, resulting in something like this:

`class, red, green, blue ...`

This structure is then fed into the machine learning classifier.

### 🔵 CHALLENGE — Field points become training data
Before running the extraction:

1. Which column is the **target/class**?
2. Which columns are the **predictors**?
3. What should happen to a field point if its satellite pixel contains NoData?

Check your answers against the resulting training table.

# Part 11 — Connect field observations to satellite pixels

This is the critical bridge between fieldwork and machine learning: each labelled observation receives satellite predictor values.

In [ ]:
# First transform the training points to the same CRS as the data
training = gdf.to_crs(median.odc.geobox.crs)

# Next get the X and Y values out of the point geometries
training_da = training.assign(x=training.geometry.x, y=training.geometry.y).to_xarray()

# Now we can use the x and y values (lon, lat) to extract values from the median composite
training_values = (
    median.sel(training_da[["x", "y"]], method="nearest").squeeze().compute().to_pandas()
)

# Join the training data with the extracted values and remove unnecessary columns
training_array = pd.concat([training["Random_Forest"], training_values], axis=1)
training_array = training_array.drop(
    columns=[
        "y",
        "x",
        "spatial_ref",
    ]
)

# Drop rows where there was no data available
training_array = training_array.dropna()

# Preview our resulting training array
training_array.head()

## Create a classifier and fit a model

We pass in simple numpy arrays to the classifier, one has the
observations (the values of the red, green, blue and so on)
while the other has the classes.

### 🔵 CHALLENGE — Predict the split
Before running the 80/20 split:

- How many observations do you expect in training?
- How many in testing?
- Why do we use `stratify=y`?
- Why are very small classes difficult to assess?

Record your estimates, then compare them with Python's output.

# Part 12 — Train/test separation and Random Forest accuracy

We hold back 20% of eligible observations. The model does not see these while learning.

**Why?** Testing a model on the same observations used to train it can make accuracy look unrealistically high.

In [ ]:
# ============================================================
# RANDOM FOREST — PROPER 80/20 HOLD-OUT ACCURACY ASSESSMENT
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

CLASS_FIELD = "Random_Forest"

# SCL is for cloud masking, not a Random Forest predictor.
predictor_names = [
    col for col in training_array.columns
    if col not in [CLASS_FIELD, "scl"]
]

print("Predictors used for Random Forest:")
print(predictor_names)

# ------------------------------------------------------------
# Exclude extremely small classes from the 80/20 evaluation ONLY.
# The original training_array is left unchanged.
# ------------------------------------------------------------
MIN_CLASS_SAMPLES = 5

class_counts = training_array[CLASS_FIELD].astype(int).value_counts()
valid_classes = class_counts[class_counts >= MIN_CLASS_SAMPLES].index

evaluation_table = training_array[
    training_array[CLASS_FIELD].astype(int).isin(valid_classes)
].copy()

excluded_classes = sorted(
    set(training_array[CLASS_FIELD].astype(int).unique())
    - set(valid_classes)
)

print("\nClasses included in 80/20 evaluation:")
print(sorted(valid_classes.tolist()))

print("\nClasses excluded from 80/20 evaluation (<5 samples):")
print(excluded_classes)

print("\nEvaluation class counts:")
print(
    evaluation_table[CLASS_FIELD]
    .astype(int)
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 80% training / 20% unseen testing
# ------------------------------------------------------------
X = evaluation_table[predictor_names]
y = evaluation_table[CLASS_FIELD].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("\n" + "=" * 60)
print("DATA SPLIT")
print("=" * 60)
print("Total evaluation observations:", len(y))
print("Training observations:", len(y_train))
print("Testing observations:", len(y_test))

# ------------------------------------------------------------
# Evaluation model
# ------------------------------------------------------------
evaluation_model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

evaluation_model.fit(X_train, y_train)
y_test_pred = evaluation_model.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
macro_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)

print("\n" + "=" * 60)
print("80/20 HOLD-OUT ACCURACY ASSESSMENT")
print("=" * 60)
print(f"Overall Accuracy : {accuracy:.4f}")
print(f"Macro F1         : {macro_f1:.4f}")
print(f"Weighted F1      : {weighted_f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred, zero_division=0))

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_test_pred, labels=labels)

print("Confusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels,
).plot(
    ax=ax,
    values_format="d",
    colorbar=False,
)
ax.set_title("Random Forest — 20% Hold-out Confusion Matrix")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# FINAL PRODUCTION MODEL
# Train using ALL valid original observations after evaluation.
# ------------------------------------------------------------
X_final = training_array[predictor_names]
y_final = training_array[CLASS_FIELD].astype(int)

model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_final, y_final)

# Compatibility with later notebook cells
classes = y_final.to_numpy()
observations = X_final.to_numpy()

print("\n" + "=" * 60)
print("FINAL PRODUCTION MODEL")
print("=" * 60)
print("Training observations:", len(X_final))
print("Predictors:", len(predictor_names))
print("Predictor names:", predictor_names)
print("Classes:", model.classes_)


## Prediction

Next we predict. Again, we need a simple numpy array, this time
just with the observations. This needs to be in long array where
the x dimension is the observation values and the y is each cell
in the original raster.

### 🟡 THINK FIRST — Wall-to-wall prediction
Before predicting the entire AOI, answer:

> If the code successfully produces a map, does that prove every classified pixel is correct?

Explain why or why not.

# Part 15 — Predict the full AOI

The trained model is now applied pixel-by-pixel across the satellite composite.

Remember: a successful code run does **not** automatically mean the map is correct.

In [ ]:
# ============================================================
# PREDICT THE FULL AOI USING THE SAME TRAINING PREDICTORS
# ============================================================

prediction_data = median[predictor_names]

stacked_arrays = (
    prediction_data
    .to_array("predictor")
    .stack(pixel=["y", "x"])
    .transpose("pixel", "predictor")
)

stacked_values = stacked_arrays.values
valid_pixels = np.all(np.isfinite(stacked_values), axis=1)

predicted = np.full(
    stacked_values.shape[0],
    np.nan,
    dtype="float32",
)

predicted[valid_pixels] = model.predict(
    stacked_values[valid_pixels]
)

array = predicted.reshape(
    len(median.y),
    len(median.x),
)

predicted_da = xr.DataArray(
    array,
    coords={"y": median.y, "x": median.x},
    dims=["y", "x"],
    name="prediction",
)

print("Prediction completed.")
print("Valid predicted pixels:", int(valid_pixels.sum()))


## Visualise our results

Here we're visualising the results along with the RGB image
and the original training data points. We're doing this using
a Python library called Folium.

In [ ]:
print(predicted_da.dtype)  # Check the dtype of your DataArray
predicted_da = predicted_da.astype('float32')  # Convert to float32

In [ ]:
# Put it all on a single interactive map
# center = [np.mean([min_lat[0], max_lat[0]]), np.mean([min_lat[1], max_lat[1]])]
# m = folium.Map(location=center, zoom_start=11)

center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]  # Assuming min_lon and max_lon are defined
m = folium.Map(location=center, zoom_start=11)



# RGB for the median
median.odc.to_rgba(vmin=0, vmax=0.3).odc.add_to(m, name="Median Composite")



<font color='blue'>4.1. - visualise the resulting machine learning prediction:

The name of your model prediction is `predicted_da` so input this before `odc.add_to`

In [ ]:
# Categorical for the predicted classes and for the training data
predicted_da.odc.add_to(m, name="Predicted")
gdf.explore(m=m, column="Random_Forest", legend=True, name="Training Data")

# Layer control
folium.LayerControl().add_to(m)

m

<font color='blue'>4.2. - write this model output map to a new file:

Input the name of your model prediction before `.odc.write_cog`

# Part 16 — Visualise and interpret the classification

Compare the classification with the satellite image and your knowledge of the landscape.

In [ ]:
predicted_da.odc.write_cog("Results/Lololo_v4.tif", overwrite=True)
# predicted_da.plot.imshow()

# Part 14 — Train the final production model

After the hold-out assessment, the production model is fitted using all valid original observations for mapping.

In [ ]:
# ============================================================
# ACCURACY NOTE
# ============================================================
# The proper accuracy assessment is now performed earlier using
# the unseen 20% hold-out test set.
#
# Do NOT use predictions on the same observations used to fit
# the final production model as the reported model accuracy.

print(f"Hold-out Overall Accuracy : {accuracy:.4f}")
print(f"Hold-out Macro F1         : {macro_f1:.4f}")
print(f"Hold-out Weighted F1      : {weighted_f1:.4f}")

print("\nThese metrics come from the unseen 20% test observations.")
print("The final production model is then fitted using all valid training observations.")


<font color='blue'>5.1. - download the file, load, explore and set new colours in QGIS:


In [ ]:
joblib.dump(model, "cordia_v.2.0.model")

## Considerations

Do the results make sense?

What are some of the limitations of the visualisation?

### Next steps

The obvious next step is to fine tune the data. Perhaps download the points for this
region of interest as well as the RGB image and add and remove points until
there is a more representative training dataset.

### 🔵 CHALLENGE — New AOI
Before running the transfer model, participants should choose or define the new AOI themselves.

Record:
- AOI/site name:
- reason for choosing it:
- expected invasive/land-cover classes:
- one reason the original model might perform differently here:

# Part 17 — Transfer the workflow to a new AOI

This demonstrates how the same trained model can be applied elsewhere. Treat transfer predictions cautiously because landscapes and spectral conditions can change.

### New AOI for your interest

Choose a new AoI and ToI based on your interests. This could be for wider Tongatapu, Eua or even Vava'u.  

In [ ]:
# 1.1 - input your four coordinates here 
# min_lat = -17.64440
# min_lon = 177.84220
# max_lat = -17.56635
# max_lon = 177.94832

# bbox_nadala = [min_lon, min_lat, max_lon, max_lat]

In [ ]:
# Define bounding box – should match your elevation mask area
aoiplot_gdf = gpd.read_file("AOI/Malekula.geojson")
aoiplot_gdf.plot(edgecolor="red", facecolor="none")
plt.title("AOI Check")
plt.show()  # Replace with actual coordinates

In [ ]:
min_lat = aoiplot_gdf['geometry'].bounds['miny'].min()
max_lat = aoiplot_gdf['geometry'].bounds['maxy'].max()
min_lon = aoiplot_gdf['geometry'].bounds['minx'].min()
max_lon = aoiplot_gdf['geometry'].bounds['maxx'].max()

In [ ]:
bbox_nadala = aoiplot_gdf.total_bounds  # returns [minx, miny, maxx, maxy]

In [ ]:
# Search for Sentinel-2 data
items = client.search(
    collections=["sentinel-2-c1-l2a"],
    bbox=bbox_nadala,
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": 15 }},
).item_collection()

print(f"Found {len(items)} items")

In [ ]:
# Load the data into an xarray Dataset
data_nadala = load(
    items,
    measurements=["red", "green", "blue", "nir08", "swir16", "scl"],
    bbox=bbox_nadala,
    chunks={"x": 800, "y": 800},
    groupby="solar_day",
)

# data_Eua

### 🟢 TRY IT — NDVI formula
Complete the formula on paper first:

**NDVI = ( ______ − ______ ) / ( ______ + ______ )**

Then find the equivalent Python calculation in the workflow.

What would you generally expect for:
- healthy vegetation:
- bare/developed surface:
- water:

# Part 9 — Create NDVI

NDVI uses red and near-infrared information to describe vegetation greenness.

**Activity:** predict whether healthy green vegetation should generally have a higher or lower NDVI than bare ground.

In [ ]:
# Mask out clouds and scale values

# Apply Sentinel-2 cloud mask
# 1: defective, 3: shadow, 9: high confidence cloud, 10: thin cirrus
mask_flags = [1, 3, 9, 10]

cloud_mask = ~data_nadala.scl.isin(mask_flags)
masked = data_nadala.where(cloud_mask)

# Apply scaling and clip to valid data, from 0 to 1
scaled_nadala = (masked.where(masked != 0) * 0.0001).clip(0, 1)

# Add some indices
scaled_nadala["ndvi"] = (scaled_nadala.nir08 - scaled_nadala.red) / (scaled_nadala.nir08 + scaled_nadala.red)
# scaled["ndwi"] = (scaled.green - scaled.nir08) / (scaled.green + scaled.nir08)

# Create a median composite, which should get rid of most of the remaining clouds
# Note that this will take a few minutes to complete

median_nadala = scaled_nadala.median("time").compute()

# median

median_nadala.odc.explore(vmin=0, vmax=0.3)

In [ ]:
# ============================================================
# PREDICT THE NEW AOI USING EXACTLY THE SAME PREDICTORS
# ============================================================

missing_new = [
    feature for feature in predictor_names
    if feature not in median_nadala.data_vars
]

if missing_new:
    raise ValueError(
        "New AOI is missing model predictors: "
        + ", ".join(missing_new)
    )

prediction_data_nadala = median_nadala[predictor_names]

stacked_arrays_nadala = (
    prediction_data_nadala
    .to_array("predictor")
    .stack(pixel=["y", "x"])
    .transpose("pixel", "predictor")
)

stacked_values_nadala = stacked_arrays_nadala.values
valid_pixels_nadala = np.all(
    np.isfinite(stacked_values_nadala),
    axis=1,
)

predicted_nadala = np.full(
    stacked_values_nadala.shape[0],
    np.nan,
    dtype="float32",
)

predicted_nadala[valid_pixels_nadala] = model.predict(
    stacked_values_nadala[valid_pixels_nadala]
)

array_nadala = predicted_nadala.reshape(
    len(median_nadala.y),
    len(median_nadala.x),
)

predicted_da = xr.DataArray(
    array_nadala,
    coords={
        "y": median_nadala.y,
        "x": median_nadala.x,
    },
    dims=["y", "x"],
    name="prediction",
)

print("New AOI prediction completed.")
print("Valid predicted pixels:", int(valid_pixels_nadala.sum()))


In [ ]:
print(predicted_da.dtype)  # Check the dtype of your DataArray
predicted_da = predicted_da.astype('float32')  # Convert to float32

# Put it all on a single interactive map
# center = [np.mean([min_lat[0], max_lat[0]]), np.mean([min_lat[1], max_lat[1]])]
# m = folium.Map(location=center, zoom_start=11)

center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]  # Assuming min_lon and max_lon are defined
m = folium.Map(location=center, zoom_start=11)

# RGB for the median
median_nadala.odc.to_rgba(vmin=0, vmax=0.3).odc.add_to(m, name="Median Composite")


In [ ]:
# Categorical for the predicted classes and for the training data
predicted_da.odc.add_to(m, name="Predicted")
gdf.explore(m=m, column="randomforest", legend=True, name="Training Data")

# Layer control
folium.LayerControl().add_to(m)

m

In [ ]:
predicted_da.odc.write_cog("Results/ML_Malekula.tif", overwrite=True)
# predicted_da.plot.imshow()

### 🔵 CHALLENGE — Design the validation
Before using the supplied accuracy code, decide:

1. What validation file will you use?
2. Was it independent of model training?
3. Which field stores the observed class?
4. How many points would you like per important class?
5. Where should validation points be distributed?

Only then set `VALIDATION_FILE`.

# Part 18 — Independent Accuracy Assessment for the New AOI

The new AOI is a **transfer prediction**. Its accuracy cannot be calculated from the classified map alone. We need **independent reference/field points inside the new AOI**.

- Original AOI: 80% training → unseen 20% testing.
- New AOI: trained model → new-area map → independent field points → observed vs predicted.

> If no independent validation points are available, do not report an accuracy value for the new AOI.

## Step 18.1 — Validation settings

Set `VALIDATION_FILE` to a point GeoJSON/Shapefile containing the observed numeric class in `Random_Forest`.

Leave it as `None` when validation data are not yet available.

In [ ]:
# Example:
# VALIDATION_FILE = "Training_Data/new_aoi_validation.geojson"

VALIDATION_FILE = None
CLASS_FIELD = "Random_Forest"

## Step 18.2 — Load independent validation points

**Checkpoint:** These points should be inside the new AOI and independent of the observations used to train the model.

In [ ]:
from pathlib import Path
import geopandas as gpd

validation_gdf = None

if VALIDATION_FILE is None:
    print("NEW AOI ACCURACY NOT CALCULATED")
    print("Independent validation points are required.")
elif not Path(VALIDATION_FILE).exists():
    print("VALIDATION FILE NOT FOUND:", VALIDATION_FILE)
else:
    validation_gdf = gpd.read_file(VALIDATION_FILE)

    if CLASS_FIELD not in validation_gdf.columns:
        raise KeyError(
            f"{CLASS_FIELD!r} not found. Available fields: "
            f"{list(validation_gdf.columns)}"
        )
    if validation_gdf.empty:
        raise ValueError("The validation file contains no observations.")
    if not all(validation_gdf.geometry.geom_type == "Point"):
        raise ValueError("Validation data must contain point geometries.")

    print("Validation points loaded:", len(validation_gdf))
    print("\nObserved class distribution:")
    print(validation_gdf[CLASS_FIELD].astype(int).value_counts().sort_index())

## Step 18.3 — Compare field classes with the new-AOI map

Each validation point is matched to the nearest classified pixel. Points outside the prediction extent or on NoData are excluded.

In [ ]:
validation_results = None

if validation_gdf is None:
    print("Skipped: no independent new-AOI validation dataset.")
else:
    prediction_crs = predicted_da.rio.crs
    if prediction_crs is None:
        raise ValueError("The new AOI prediction has no CRS.")

    validation_gdf = validation_gdf.to_crs(prediction_crs)

    xmin, xmax = float(predicted_da.x.min()), float(predicted_da.x.max())
    ymin, ymax = float(predicted_da.y.min()), float(predicted_da.y.max())

    records = []

    for idx, row in validation_gdf.iterrows():
        x, y = row.geometry.x, row.geometry.y

        if not (xmin <= x <= xmax and ymin <= y <= ymax):
            continue

        value = predicted_da.sel(x=x, y=y, method="nearest").item()

        if np.isfinite(value):
            observed_class = int(row[CLASS_FIELD])
            predicted_class = int(value)

            records.append({
                "source_index": idx,
                "observed": observed_class,
                "predicted": predicted_class,
                "correct": observed_class == predicted_class,
            })

    validation_results = pd.DataFrame(records)

    print("Total validation points:", len(validation_gdf))
    print("Valid comparison points:", len(validation_results))
    print(
        "Excluded/outside/NoData:",
        len(validation_gdf) - len(validation_results)
    )

    if not validation_results.empty:
        display(validation_results.head())

## Step 18.4 — New AOI transfer accuracy

The following reports Overall Accuracy, Macro F1, Weighted F1 and class-level precision/recall/F1.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

if validation_results is None or validation_results.empty:
    print(
        "Accuracy cannot be calculated: no valid independent "
        "new-AOI reference comparisons are available."
    )
else:
    observed_new = validation_results["observed"].to_numpy()
    predicted_new = validation_results["predicted"].to_numpy()

    new_aoi_accuracy = accuracy_score(observed_new, predicted_new)
    new_aoi_macro_f1 = f1_score(
        observed_new, predicted_new,
        average="macro", zero_division=0
    )
    new_aoi_weighted_f1 = f1_score(
        observed_new, predicted_new,
        average="weighted", zero_division=0
    )

    print("=" * 60)
    print("NEW AOI — INDEPENDENT TRANSFER ACCURACY")
    print("=" * 60)
    print("Valid validation points :", len(validation_results))
    print(f"Overall Accuracy        : {new_aoi_accuracy:.4f}")
    print(f"Macro F1                : {new_aoi_macro_f1:.4f}")
    print(f"Weighted F1             : {new_aoi_weighted_f1:.4f}")

    print("\nClassification Report:\n")
    print(
        classification_report(
            observed_new,
            predicted_new,
            zero_division=0
        )
    )

## Step 18.5 — New AOI confusion matrix

Rows are **observed field classes** and columns are **predicted classes**. Off-diagonal values show class-to-class errors.

In [ ]:
if validation_results is None or validation_results.empty:
    print("Confusion matrix skipped: independent validation data are required.")
else:
    labels_new = sorted(
        np.unique(np.concatenate([observed_new, predicted_new]))
    )

    cm_new = confusion_matrix(
        observed_new,
        predicted_new,
        labels=labels_new
    )

    print("Confusion Matrix:")
    print(cm_new)

    fig, ax = plt.subplots(figsize=(9, 8))
    ConfusionMatrixDisplay(
        confusion_matrix=cm_new,
        display_labels=labels_new
    ).plot(
        ax=ax,
        values_format="d",
        colorbar=False
    )

    ax.set_title("New AOI — Independent Validation Confusion Matrix")
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("Observed Field Class")
    plt.tight_layout()
    plt.show()

## Step 18.6 — Participant interpretation activity

Answer:

1. What is the new AOI Overall Accuracy?
2. What is its Macro F1?
3. Which class has the highest recall?
4. Which class has the lowest recall?
5. Which classes are most frequently confused?
6. Are any validation classes represented by too few observations?
7. Is transfer accuracy higher or lower than the original 80/20 accuracy?
8. What could explain the difference?
9. Where should more field validation be collected?

A drop in transfer accuracy can reflect differences in vegetation, land use, seasonality, spectral conditions, training representation or class definitions—not necessarily a coding error.

# Part 19 — Participant model-diagnostics activity

Use your hold-out classification report and confusion matrix to answer:

1. What was the overall accuracy?
2. Which class had the highest recall?
3. Which class had the lowest recall?
4. Which two classes were most frequently confused?
5. Does the class with the most observations also perform best?
6. Are any classes represented by too few observations to assess confidently?
7. Where would you collect additional field observations?
8. Would you trust this map immediately for operational decision-making? Why or why not?

Write your answers in the Markdown cell below.

### My answers

1. Overall accuracy:
2. Highest-recall class:
3. Lowest-recall class:
4. Main confusion:
5. Effect of class balance:
6. Undersampled classes:
7. Priority locations/classes for more fieldwork:
8. Would I trust the map operationally? Why/why not:

# Part 20 — Beginner challenge: change one thing, not everything

A good modelling experiment changes **one clearly defined component** and compares the result.

Choose one:

- collect/add more observations for a weak class;
- add another scientifically relevant spectral predictor;
- change the AOI;
- compare a different date range;
- investigate a confusion pair.

Record:

**What did I change?**  
**Why did I change it?**  
**What metric will tell me whether it helped?**  
**Did the map improve visually as well as statistically?**

This is the beginning of iterative model improvement.

# Participant Capstone — Build your own mini experiment

Your group now chooses **one** question to investigate without being given the complete solution.

Examples:
- Does changing the date range alter invasive-species performance?
- Does adding another vegetation/spectral index help a weak class?
- Does adding more field observations improve recall?
- Does the model transfer well to another AOI?
- Which class is most frequently confused with an invasive species?

## Your experiment

**Question:**  

**Hypothesis — what do you expect to happen?**  

**One thing you will change:**  

**Things you will keep the same:**  

**Metric(s) you will compare:**  

**Result:**  

**Did the result support your hypothesis?**  

**What would you do next?**

## Facilitator checkpoint

Before presenting your result, your group should be able to explain:

**Input data → change made → model output → accuracy evidence → map evidence → conclusion**

A result such as *"accuracy increased"* is not enough. Identify **which classes improved or worsened** and whether the mapped pattern makes sense geographically.

# Final workshop checkpoint

By the end of this notebook you should be able to describe the complete workflow in your own words:

**Field observations → AOI → Sentinel-2 → cloud masking → spectral predictors → training table → 80/20 evaluation → Random Forest → confusion matrix → final production model → classification map → interpretation → improvement**

The key lesson is:

> **A Random Forest map is not automatically correct because the code ran successfully.**

The quality of the final map depends on the quality and representativeness of the field observations, the suitability of the satellite predictors, the evaluation design, and careful interpretation of model errors.